# DQN

### 1. Imports, Random Seed, Load RL Environment, Hyperparameters

In [ ]:
import os
import random
import pickle
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from collections import deque
from sklearn.metrics.pairwise import cosine_similarity
warnings.filterwarnings("ignore")

### 2. Random Seed

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

### 3. Device

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("="*60)
print("Running Device :", device)
print("="*60)

Running Device : cuda


### 4. Load RL Environment

In [ ]:
RL_FILE = "rl_environment.pkl"

if not os.path.exists(RL_FILE):

    raise FileNotFoundError(
        "rl_environment.pkl not found.\n"
        "Upload the file before running."
    )

with open(RL_FILE, "rb") as f:

    folds = pickle.load(f)

print("RL Environment Loaded Successfully")
print("Number of Folds :", len(folds))
print()

RL Environment Loaded Successfully
Number of Folds : 5



### 5. Verify Each Fold

In [ ]:
for fold in folds:
    print("-"*50)
    print("Fold :", fold["fold"])
    print("Training States :", fold["states_train"].shape)
    print("Testing States  :", fold["states_test"].shape)
    print("Training Labels :", fold["labels_train"].shape)
    print("Testing Labels  :", fold["labels_test"].shape)
    print("State Size      :", fold["state_size"])
    print("Action Size     :", fold["action_size"])
print("-"*50)

--------------------------------------------------
Fold : 1
Training States : (334, 15)
Testing States  : (84, 15)
Training Labels : (334,)
Testing Labels  : (84,)
State Size      : 15
Action Size     : 2
--------------------------------------------------
Fold : 2
Training States : (334, 15)
Testing States  : (84, 15)
Training Labels : (334,)
Testing Labels  : (84,)
State Size      : 15
Action Size     : 2
--------------------------------------------------
Fold : 3
Training States : (334, 15)
Testing States  : (84, 15)
Training Labels : (334,)
Testing Labels  : (84,)
State Size      : 15
Action Size     : 2
--------------------------------------------------
Fold : 4
Training States : (335, 15)
Testing States  : (83, 15)
Training Labels : (335,)
Testing Labels  : (83,)
State Size      : 15
Action Size     : 2
--------------------------------------------------
Fold : 5
Training States : (335, 15)
Testing States  : (83, 15)
Training Labels : (335,)
Testing Labels  : (83,)
State Size      

### 6. State Informatiom

In [ ]:
STATE_SIZE = folds[0]["state_size"]
NUM_FEATURES = STATE_SIZE

### 7. Action Space

In [ ]:
# Actions
# 1. Increase
# 2. Decrease
# 3. Hold
# Number of Actions = 3 × Number of Features

ACTION_SIZE = NUM_FEATURES * 3
print("Input Features :", NUM_FEATURES)
print("Total Actions  :", ACTION_SIZE)

# Action Definitions
ACTION_INCREASE = 1
ACTION_DECREASE = -1
ACTION_HOLD = 0
DELTA = 0.05

Input Features : 15
Total Actions  : 45


### 8. DQN Hyperparameters

In [ ]:
NUM_EPISODES = 300
MAX_STEPS = 300   #300
GAMMA = 0.99 #0.99
LEARNING_RATE = 0.001
EPSILON_START = 1.0
EPSILON_MIN = 0.01
EPSILON_DECAY = 0.995
BATCH_SIZE = 64
MEMORY_SIZE = 10000
TARGET_UPDATE = 10

### 9. Network Architecture

In [ ]:
HIDDEN_LAYER_1 = 128
HIDDEN_LAYER_2 = 64

### 10. Training Storage

In [ ]:
trained_models = []
training_rewards = []
training_losses = []
training_history = []
evaluation_results = []

### 11. Display Settings

In [ ]:
print("EHR-DQN CONFIGURATION")
print("="*60)

print(f"Input Features         : {STATE_SIZE}")
print(f"Action Space           : {ACTION_SIZE}")
print(f"Episodes               : {NUM_EPISODES}")
print(f"Maximum Steps          : {MAX_STEPS}")
print(f"Replay Memory          : {MEMORY_SIZE}")
print(f"Mini Batch Size        : {BATCH_SIZE}")
print(f"Learning Rate          : {LEARNING_RATE}")
print(f"Discount Factor        : {GAMMA}")
print(f"Target Update          : {TARGET_UPDATE}")
print(f"Epsilon Start          : {EPSILON_START}")
print(f"Epsilon Minimum        : {EPSILON_MIN}")
print(f"Epsilon Decay          : {EPSILON_DECAY}")
print(f"State Modification Δ   : {DELTA}")
print(f"Device                 : {device}")

EHR-DQN CONFIGURATION
Input Features         : 15
Action Space           : 45
Episodes               : 300
Maximum Steps          : 300
Replay Memory          : 10000
Mini Batch Size        : 64
Learning Rate          : 0.001
Discount Factor        : 0.99
Target Update          : 10
Epsilon Start          : 1.0
Epsilon Minimum        : 0.01
Epsilon Decay          : 0.995
State Modification Δ   : 0.05
Device                 : cuda


### 12. Disease Prediction Environment

In [ ]:
class DiseasePredictionEnvironment:

    def __init__(self, states, labels):
        self.original_states = np.array(states, dtype=np.float32)
        self.labels = np.array(labels, dtype=np.int64)
        self.num_samples = len(self.original_states)
        self.num_features = self.original_states.shape[1]
        self.reset()


    # Reset Environment
    def reset(self):
        self.current_index = np.random.randint(0, self.num_samples)
        self.current_state = self.original_states[self.current_index].copy()
        self.steps = 0
        return self.current_state.copy()


    # Modify Feature
    def modify_state(self, state, action):

        new_state = state.copy()
        feature = action // 3
        operation = action % 3

        if operation == ACTION_INCREASE:
            new_state[feature] += DELTA
        elif operation == ACTION_DECREASE:
            new_state[feature] -= DELTA
        elif operation == ACTION_HOLD:
            pass
        return new_state


    # Find Most Similar Patient
    def nearest_patient(self, modified_state):
        similarities = cosine_similarity(modified_state.reshape(1,-1), self.original_states)[0]
        best_index = np.argmax(similarities)
        return best_index, similarities[best_index]


    # Step
    def step(self, action):
        self.steps += 1
        modified_state = self.modify_state(self.current_state, action)
        next_index, similarity = self.nearest_patient(modified_state)
        next_state = self.original_states[next_index].copy()
        label = self.labels[next_index]

        # Reward Function
        if label == 1:
            reward = 10.0
            done = True
        else:
            reward = -1.0
            done = False
        if self.steps >= MAX_STEPS:
            done = True

        self.current_index = next_index
        self.current_state = next_state.copy()

        info = {"matched_patient": next_index,
                "label": int(label),
                "similarity": float(similarity),
                "steps": self.steps}

        return (next_state, reward, done, info)



    # Sample Random Action
    def sample_action(self):
        return random.randint(0, ACTION_SIZE - 1)



# Verify Environment
print("Testing Environment")
sample_fold = folds[0]
env = DiseasePredictionEnvironment(sample_fold["states_train"], sample_fold["labels_train"])
state = env.reset()

print("Initial State Shape :", state.shape)
print()

action = env.sample_action()
print("Random Action :", action)

next_state, reward, done, info = env.step(action)
print()
print("Reward :", reward)
print("Done :", done)

print("Next State Shape :", next_state.shape)
print()

print("Matched Patient :", info["matched_patient"])
print("Similarity :", round(info["similarity"],4))
print("Matched Label :", info["label"])
print()

print("="*60)
print("Environment Ready")
print("="*60)

Testing Environment
Initial State Shape : (15,)

Random Action : 40

Reward : 10.0
Done : True
Next State Shape : (15,)

Matched Patient : 270
Similarity : 0.9998
Matched Label : 1

Environment Ready


### 13. Experience Replay Memory

In [ ]:
class ReplayMemory:

    def __init__(self, capacity):
        self.capacity = capacity
        self.memory = deque(maxlen=capacity)


    # Store Experience
    def push(self, state, action, reward, next_state, done):
        experience = (np.array(state, dtype=np.float32),
                      int(action),
                      float(reward),
                      np.array(next_state, dtype=np.float32),
                      bool(done))
        self.memory.append(experience)


    # Sample Mini Batch
    def sample(self,batch_size):
        batch = random.sample(self.memory, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        states = torch.FloatTensor(np.array(states)).to(device)
        actions = torch.LongTensor(np.array(actions)).unsqueeze(1).to(device)
        rewards = torch.FloatTensor(np.array(rewards)).unsqueeze(1).to(device)
        next_states = torch.FloatTensor(np.array(next_states)).to(device)
        dones = torch.FloatTensor(np.array(dones)).unsqueeze(1).to(device)

        return (states, actions, rewards, next_states, dones)


    # Current Memory Size
    def __len__(self):
        return len(self.memory)

    # Check Enough Samples
    def ready(self):
        return len(self.memory) >= BATCH_SIZE

    # Clear Memory
    def clear(self):
        self.memory.clear()

    # Save Replay Buffer
    def save(self, filename):
        with open(filename, "wb") as f:
            pickle.dump(list(self.memory),f)


    # Load Replay Buffer
    def load(self, filename):
        with open(filename, "rb") as f:
            data = pickle.load(f)

        self.memory = deque(data, maxlen=self.capacity)



# Create Replay Memory
replay_memory = ReplayMemory(MEMORY_SIZE)

print("="*60)
print("Replay Memory Created")
print("="*60)
print("Capacity :", MEMORY_SIZE)
print("Current Size :", len(replay_memory))
print("="*60)


# Test Replay Memory
print("\nTesting Replay Memory...\n")
sample_state = np.random.rand(STATE_SIZE).astype(np.float32)
sample_next_state = np.random.rand(STATE_SIZE).astype(np.float32)
replay_memory.push(sample_state, 5, 1.0, sample_next_state, False)
print("Memory Size After Insert :", len(replay_memory))

if replay_memory.ready():
    batch = replay_memory.sample(BATCH_SIZE)
    print("Mini-batch Ready")
else:
    print("Waiting for more experiences...")

print()
print("="*60)


Replay Memory Created
Capacity : 10000
Current Size : 0

Testing Replay Memory...

Memory Size After Insert : 1
Waiting for more experiences...



### 14. Deep Q Network (Policy Network + Target Network)

In [ ]:
class DeepQNetwork(nn.Module):

    def __init__(self, input_dim, output_dim):
        super(DeepQNetwork, self).__init__()
        self.model = nn.Sequential(

            # Hidden Layer 1
            nn.Linear(input_dim, 128),
            nn.ReLU(),

            # Hidden Layer 2
            nn.Linear(128, 256),
            nn.ReLU(),

            # Hidden Layer 3
            nn.Linear(256, 128),
            nn.ReLU(),

            # Output Layer
            nn.Linear(128, output_dim)

        )


    # Forward Pass
    def forward(self, x):
        return self.model(x)


# Create Policy Network
policy_network = DeepQNetwork(input_dim=STATE_SIZE, output_dim=ACTION_SIZE).to(device)

# Create Target Network
target_network = DeepQNetwork(input_dim=STATE_SIZE, output_dim=ACTION_SIZE).to(device)

# Copy Initial Weights
target_network.load_state_dict(policy_network.state_dict())
target_network.eval()

# Loss Function
criterion = nn.SmoothL1Loss()

# Optimizer
optimizer = optim.Adam(policy_network.parameters(), lr=LEARNING_RATE)

# Soft Target Update Function
def soft_update(policy_net, target_net, tau=0.005):

    for target_param, policy_param in zip(target_net.parameters(), policy_net.parameters()):
        target_param.data.copy_(tau * policy_param.data + (1.0 - tau) * target_param.data)

# Hard Target Update Function
def hard_update(policy_net, target_net):
    target_net.load_state_dict(policy_net.state_dict())

# Save Model
def save_model(model, filename):
    torch.save(model.state_dict(), filename)

# Load Model
def load_model(model, filename):
    model.load_state_dict(torch.load(filename, map_location=device))
    model.eval()

# Display Model Summary
print("=" * 60)
print("Policy Network")
print("=" * 60)
print(policy_network)
print()
print("=" * 60)
print("Target Network Created")
print("=" * 60)


# Parameter Count
total_parameters = sum(p.numel() for p in policy_network.parameters())
trainable_parameters = sum(p.numel() for p in policy_network.parameters() if p.requires_grad)

print("Total Parameters      :", total_parameters)
print("Trainable Parameters  :", trainable_parameters)
print()


# Test Froward Pass
dummy_state = torch.randn(1, STATE_SIZE).to(device)

with torch.no_grad():
    q_values = policy_network(dummy_state)



print("Example Q Values")
print(q_values)
print()
print("Output Shape :", q_values.shape)
print()
print("=" * 60)
print("Block 4 Completed Successfully")
print("=" * 60)

Policy Network
DeepQNetwork(
  (model): Sequential(
    (0): Linear(in_features=15, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): ReLU()
    (6): Linear(in_features=128, out_features=45, bias=True)
  )
)

Target Network Created
Total Parameters      : 73773
Trainable Parameters  : 73773

Example Q Values
tensor([[-0.0374,  0.0699,  0.0514, -0.0849, -0.0174,  0.0952, -0.0828,  0.0671,
          0.0430, -0.0109, -0.0477,  0.0710,  0.0226, -0.0764,  0.0303, -0.0788,
          0.0706,  0.0034, -0.0986,  0.1225,  0.0672,  0.0198, -0.0107, -0.0542,
          0.0531, -0.0470, -0.0184, -0.0968, -0.0341, -0.0302, -0.0295, -0.0973,
         -0.0268,  0.0062,  0.0439,  0.1042,  0.0093,  0.0542, -0.0772,  0.0224,
         -0.0545,  0.0579, -0.0468,  0.0349, -0.0809]], device='cuda:0')

Output Shape : torch.Size([1, 45])

Block 4 Completed Successfully


### 15. DQN Agent

In [ ]:
class DQNAgent:

    def __init__(self):
        self.policy_network = DeepQNetwork(STATE_SIZE, ACTION_SIZE).to(device)
        self.target_network = DeepQNetwork(STATE_SIZE, ACTION_SIZE).to(device)
        self.target_network.load_state_dict(self.policy_network.state_dict())
        self.target_network.eval()
        self.optimizer = optim.Adam(self.policy_network.parameters(), lr=LEARNING_RATE)
        self.criterion = nn.SmoothL1Loss()
        self.memory = ReplayMemory(MEMORY_SIZE)
        self.gamma = GAMMA
        self.batch_size = BATCH_SIZE
        self.epsilon = EPSILON_START
        self.epsilon_min = EPSILON_MIN
        self.epsilon_decay = EPSILON_DECAY


    def act(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, ACTION_SIZE - 1)

        state = torch.FloatTensor(state).unsqueeze(0).to(device)

        with torch.no_grad():
            q_values = self.policy_network(state)

        return torch.argmax(q_values).item()


    def remember(self, state, action, reward, next_state, done):
        self.memory.push(state, action, reward, next_state, done)


    def replay(self):
        if len(self.memory) < self.batch_size:
            return None

        states, actions, rewards, next_states, dones = self.memory.sample(self.batch_size)
        current_q = self.policy_network(states).gather(1, actions)

        with torch.no_grad():
            next_q = self.target_network(next_states).max(1)[0].unsqueeze(1)
            target_q = rewards + (1 - dones) * self.gamma * next_q

        loss = self.criterion(current_q, target_q)
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.policy_network.parameters(), 1.0)
        self.optimizer.step()

        return loss.item()


    def update_target(self):
        self.target_network.load_state_dict(self.policy_network.state_dict())


    def decay_epsilon(self):
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay
            self.epsilon = max(self.epsilon,  self.epsilon_min)


    def save(self, filename):
        torch.save(self.policy_network.state_dict(), filename)


    def load(self, filename):
        self.policy_network.load_state_dict(torch.load(filename, map_location=device))
        self.update_target()

print("="*60)
print("DQN Agent Ready")
print("="*60)


DQN Agent Ready


### 16. Training One Episode

In [ ]:
def train_one_episode(agent, env):
    """
    Train the DQN agent for one episode.

    Returns
    -------
    episode_reward : float
    mean_loss : float
    steps : int
    """

    state = env.reset()
    done = False
    episode_reward = 0.0
    losses = []
    steps = 0

    while not done and steps < MAX_STEPS:

        # Select Action (ε-greedy)
        action = agent.act(state)

        # Environment Step
        next_state, reward, done, info = env.step(action)

        # Store Transition
        agent.remember(state, action, reward, next_state, done)

        # Learn from Replay Buffer
        loss = agent.replay()

        if loss is not None:
            losses.append(loss)

        state = next_state
        episode_reward += reward
        steps += 1

    # Update Target Network
    agent.update_target()

    # Decay ε
    agent.decay_epsilon()

    if len(losses) > 0:
        mean_loss = sum(losses) / len(losses)
    else:
        mean_loss = 0.0

    return episode_reward, mean_loss, steps

# QUICK SANITY TEST
if __name__ == "__main__":

    sample_fold = folds[0]

    env = DiseasePredictionEnvironment(sample_fold["states_train"],
                                       sample_fold["labels_train"])

    agent = DQNAgent()
    reward, loss, steps = train_one_episode(agent, env)

    print("=" * 60)
    print("Episode Finished")
    print("=" * 60)
    print("Reward :", reward)
    print("Loss   :", loss)
    print("Steps  :", steps)
    print("Epsilon:", agent.epsilon)
    print("=" * 60)


Episode Finished
Reward : 10.0
Loss   : 0.0
Steps  : 1
Epsilon: 0.995


### 17. Train One Fold

In [ ]:
def train_one_fold(fold_data, fold_number=None):

    if fold_number is None:
        fold_number = fold_data["fold"]

    print("="*60)
    print(f"Training Fold {fold_number}")
    print("="*60)

    env = DiseasePredictionEnvironment(fold_data["states_train"], fold_data["labels_train"])
    agent = DQNAgent()

    reward_history = []
    loss_history = []
    step_history = []

    best_reward = float("-inf")

    for episode in range(NUM_EPISODES):
        reward, loss, steps = train_one_episode(agent, env)
        reward_history.append(reward)
        loss_history.append(loss)
        step_history.append(steps)

        # Periodic Target Sync
        if (episode + 1) % TARGET_UPDATE == 0:
            agent.update_target()

        if reward > best_reward:
            best_reward = reward

        if (episode + 1) % 10 == 0 or episode == 0:
            avg_reward = sum(reward_history[-10:]) / min(10, len(reward_history))
            avg_loss = sum(loss_history[-10:]) / min(10, len(loss_history))

            print(f"Episode {episode+1:3d}/{NUM_EPISODES} | "
                  f"Reward {reward:7.2f} | "
                  f"AvgReward {avg_reward:7.2f} | "
                  f"Loss {avg_loss:.5f} | "
                  f"Epsilon {agent.epsilon:.4f}")

    history = {"fold": fold_number,
               "reward_history": reward_history,
               "loss_history": loss_history,
               "step_history": step_history,
               "best_reward": best_reward}

    print("-"*60)
    print(f"Fold {fold_number} Completed")
    print(f"Best Reward : {best_reward:.2f}")
    print(f"Final Epsilon : {agent.epsilon:.4f}")
    print("-"*60)

    return agent, history

# Sanity Test
if __name__ == "__main__":
    agent, history = train_one_fold(folds[0])

    print("="*60)
    print("History Keys")
    print(history.keys())
    print("Episodes :", len(history["reward_history"]))
    print("="*60)


Training Fold 1
Episode   1/300 | Reward   10.00 | AvgReward   10.00 | Loss 0.00000 | Epsilon 0.9950
Episode  10/300 | Reward   10.00 | AvgReward -145.00 | Loss 0.18465 | Epsilon 0.9511
Episode  20/300 | Reward   10.00 | AvgReward -114.00 | Loss 0.19753 | Epsilon 0.9046
Episode  30/300 | Reward -300.00 | AvgReward  -83.00 | Loss 0.24401 | Epsilon 0.8604
Episode  40/300 | Reward   10.00 | AvgReward -114.00 | Loss 0.23273 | Epsilon 0.8183
Episode  50/300 | Reward   10.00 | AvgReward  -83.00 | Loss 0.30264 | Epsilon 0.7783
Episode  60/300 | Reward   10.00 | AvgReward  -83.00 | Loss 0.21008 | Epsilon 0.7403
Episode  70/300 | Reward -300.00 | AvgReward -176.00 | Loss 0.14553 | Epsilon 0.7041
Episode  80/300 | Reward   10.00 | AvgReward -176.00 | Loss 0.14169 | Epsilon 0.6696
Episode  90/300 | Reward   10.00 | AvgReward  -52.00 | Loss 0.16547 | Epsilon 0.6369
Episode 100/300 | Reward -300.00 | AvgReward -176.00 | Loss 0.12797 | Epsilon 0.6058
Episode 110/300 | Reward   10.00 | AvgReward -145

### 18. Train All 5 Fold

In [ ]:
all_agents = []
all_histories = []

overall_rewards = []
overall_losses = []

print("="*70)
print("STARTING 5-FOLD DQN TRAINING")
print("="*70)

for fold_data in folds:
    fold_number = fold_data["fold"]
    agent, history = train_one_fold(fold_data, fold_number=fold_number)

    all_agents.append(agent)
    all_histories.append(history)

    overall_rewards.append(history["reward_history"])
    overall_losses.append(history["loss_history"])

    print(f"Fold {fold_number} training completed.")
    print()

print("="*70)
print("ALL FOLDS TRAINED")
print("="*70)

# Summary
for history in all_histories:
    rewards = history["reward_history"]
    losses = history["loss_history"]

    avg_reward = sum(rewards) / len(rewards)
    avg_loss = sum(losses) / len(losses)

    print(f"Fold {history['fold']} | "
          f"Avg Reward: {avg_reward:.2f} | "
          f"Best Reward: {history['best_reward']:.2f} | "
          f"Avg Loss: {avg_loss:.6f}")

print("="*70)

training_results = {"agents": all_agents,
                    "histories": all_histories,
                    "reward_history": overall_rewards,
                    "loss_history": overall_losses}

print("Training results object created.")
print("Keys:", list(training_results.keys()))

STARTING 5-FOLD DQN TRAINING
Training Fold 1
Episode   1/300 | Reward   10.00 | AvgReward   10.00 | Loss 0.00000 | Epsilon 0.9950
Episode  10/300 | Reward -300.00 | AvgReward -114.00 | Loss 0.19059 | Epsilon 0.9511
Episode  20/300 | Reward   10.00 | AvgReward  -83.00 | Loss 0.21583 | Epsilon 0.9046
Episode  30/300 | Reward   10.00 | AvgReward  -52.00 | Loss 0.19804 | Epsilon 0.8604
Episode  40/300 | Reward   10.00 | AvgReward -176.00 | Loss 0.12348 | Epsilon 0.8183
Episode  50/300 | Reward   10.00 | AvgReward -145.00 | Loss 0.14937 | Epsilon 0.7783
Episode  60/300 | Reward   10.00 | AvgReward -145.00 | Loss 0.13936 | Epsilon 0.7403
Episode  70/300 | Reward   10.00 | AvgReward  -83.00 | Loss 0.12031 | Epsilon 0.7041
Episode  80/300 | Reward -300.00 | AvgReward -114.00 | Loss 0.11271 | Epsilon 0.6696
Episode  90/300 | Reward -300.00 | AvgReward  -83.00 | Loss 0.19084 | Epsilon 0.6369
Episode 100/300 | Reward   10.00 | AvgReward -176.00 | Loss 0.13502 | Epsilon 0.6058
Episode 110/300 | Re

### 19. Save Models and Training History

In [ ]:
# Create Output Directories
MODEL_DIR = "saved_models"
HISTORY_DIR = "training_history"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(HISTORY_DIR, exist_ok=True)

print("=" * 70)
print("Saving Trained Models")
print("=" * 70)

saved_model_paths = []
# Save Each Fold Model
for i, agent in enumerate(all_agents):
    fold_number = i + 1
    model_path = os.path.join(MODEL_DIR, f"dqn_fold_{fold_number}.pth")
    torch.save(agent.policy_network.state_dict(), model_path)
    saved_model_paths.append(model_path)
    print(f"Fold {fold_number} model saved")
    print(f"  {model_path}")

print("=" * 70)

# Save Training History
history_file = os.path.join(HISTORY_DIR, "training_history.pkl")

with open(history_file, "wb") as f:
    pickle.dump(all_histories, f)

print("Training history saved")
print(history_file)

# Save Combined Training Results
results_file = os.path.join(HISTORY_DIR, "training_results.pkl")

serializable_results = {"reward_history": overall_rewards,
                        "loss_history": overall_losses,
                        "histories": all_histories,
                        "model_paths": saved_model_paths}

with open(results_file, "wb") as f:
    pickle.dump(serializable_results, f)

print("Combined training results saved")
print(results_file)

# Save Configuration
config = {"state_size": STATE_SIZE,
          "action_size": ACTION_SIZE,
          "episodes": NUM_EPISODES,
          "batch_size": BATCH_SIZE,
          "gamma": GAMMA,
          "learning_rate": LEARNING_RATE,
          "epsilon_start": EPSILON_START,
          "epsilon_min": EPSILON_MIN,
          "epsilon_decay": EPSILON_DECAY,
          "memory_size": MEMORY_SIZE,
          "target_update": TARGET_UPDATE}

config_file = os.path.join(HISTORY_DIR, "config.pkl")

with open(config_file, "wb") as f:
    pickle.dump(config, f)

print("Configuration saved")
print(config_file)

print("=" * 70)
print("Saved Files Summary")
print("=" * 70)

for path in saved_model_paths:
    print(path)

print(history_file)
print(results_file)
print(config_file)

Saving Trained Models
Fold 1 model saved
  saved_models/dqn_fold_1.pth
Fold 2 model saved
  saved_models/dqn_fold_2.pth
Fold 3 model saved
  saved_models/dqn_fold_3.pth
Fold 4 model saved
  saved_models/dqn_fold_4.pth
Fold 5 model saved
  saved_models/dqn_fold_5.pth
Training history saved
training_history/training_history.pkl
Combined training results saved
training_history/training_results.pkl
Configuration saved
training_history/config.pkl
Saved Files Summary
saved_models/dqn_fold_1.pth
saved_models/dqn_fold_2.pth
saved_models/dqn_fold_3.pth
saved_models/dqn_fold_4.pth
saved_models/dqn_fold_5.pth
training_history/training_history.pkl
training_history/training_results.pkl
training_history/config.pkl


### 20. Evaluation

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score)

all_metrics = []

print("="*70)
print("EVALUATING ALL FOLDS")
print("="*70)

for fold_idx, fold_data in enumerate(folds):
    print(f"\nEvaluating Fold {fold_idx+1}")

    agent = all_agents[fold_idx]
    agent.policy_network.eval()

    X_test = np.array(fold_data["states_test"], dtype=np.float32)
    y_test = np.array(fold_data["labels_test"], dtype=np.int64)

    predictions = []
    probabilities = []

    with torch.no_grad():
        for state in X_test:
            state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
            q_values = agent.policy_network(state_tensor)
            probs = torch.softmax(q_values, dim=1)

            # Binary prediction using first two outputs
            pred = torch.argmax(q_values[:, :2], dim=1).item()

            predictions.append(pred)
            probabilities.append(probs[0,1].item())

    acc = accuracy_score(y_test, predictions)
    prec = precision_score(y_test, predictions, zero_division=0)
    rec = recall_score(y_test, predictions, zero_division=0)
    f1 = f1_score(y_test, predictions, zero_division=0)

    cm = confusion_matrix(y_test, predictions)

    if cm.shape == (2,2):
        tn, fp, fn, tp = cm.ravel()
        specificity = tn / (tn + fp) if (tn+fp)>0 else 0
    else:
        specificity = 0

    try:
        auc = roc_auc_score(y_test, probabilities)
    except:
        auc = float("nan")

    metrics = {"Fold": fold_idx+1,
               "Accuracy": acc,
               "Precision": prec,
               "Recall": rec,
               "F1": f1,
               "Specificity": specificity,
               "ROC_AUC": auc}

    all_metrics.append(metrics)
    print(metrics)
    print("Confusion Matrix")
    print(cm)

results_df = pd.DataFrame(all_metrics)

print("\n"+"="*70)
print("AVERAGE PERFORMANCE")
print("="*70)

print(results_df.mean(numeric_only=True))

results_df.to_csv("evaluation_results.csv", index=False)

print("\nSaved evaluation_results.csv")
print("="*70)


EVALUATING ALL FOLDS

Evaluating Fold 1
{'Fold': 1, 'Accuracy': 0.6309523809523809, 'Precision': 0.6666666666666666, 'Recall': 0.4878048780487805, 'F1': 0.5633802816901409, 'Specificity': np.float64(0.7674418604651163), 'ROC_AUC': np.float64(0.40612592172433354)}
Confusion Matrix
[[33 10]
 [21 20]]

Evaluating Fold 2
{'Fold': 2, 'Accuracy': 0.25, 'Precision': 0.3191489361702128, 'Recall': 0.32608695652173914, 'F1': 0.3225806451612903, 'Specificity': np.float64(0.15789473684210525), 'ROC_AUC': np.float64(0.2511441647597254)}
Confusion Matrix
[[ 6 32]
 [31 15]]

Evaluating Fold 3
{'Fold': 3, 'Accuracy': 0.6666666666666666, 'Precision': 0.9, 'Recall': 0.4090909090909091, 'F1': 0.5625, 'Specificity': np.float64(0.95), 'ROC_AUC': np.float64(0.2670454545454546)}
Confusion Matrix
[[38  2]
 [26 18]]

Evaluating Fold 4
{'Fold': 4, 'Accuracy': 0.5783132530120482, 'Precision': 0.573170731707317, 'Recall': 1.0, 'F1': 0.7286821705426356, 'Specificity': np.float64(0.027777777777777776), 'ROC_AUC': n

### 21. EXPORT RESULTS & INFERENCE DEMO

In [ ]:
OUTPUT_DIR = "deployment_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save metrics to Excel
csv_file = "evaluation_results.csv"

if os.path.exists(csv_file):
    df = pd.read_csv(csv_file)
    xlsx_path = os.path.join(OUTPUT_DIR, "evaluation_results.xlsx")
    with pd.ExcelWriter(xlsx_path) as writer:
        df.to_excel(writer, index=False)
    print(f"Saved: {xlsx_path}")
else:
    print("evaluation_results.csv not found.")

# Simple inference function

def predict_patient(agent, patient_features):
    """
    Predict Q-values for one patient.

    Parameters
    ----------
    agent : trained DQNAgent
    patient_features : array-like shape (STATE_SIZE,)

    Returns
    -------
    dict
    """
    state = np.asarray(patient_features, dtype=np.float32)

    if state.shape[0] != STATE_SIZE:
        raise ValueError(f"Expected {STATE_SIZE} features, got {state.shape[0]}")

    agent.policy_network.eval()

    with torch.no_grad():
        tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
        q_values = agent.policy_network(tensor)
        probs = torch.softmax(q_values, dim=1)
        action = torch.argmax(q_values).item()

    return {"recommended_action": action,
            "q_values": q_values.cpu().numpy().flatten(),
            "action_probability": probs.cpu().numpy().flatten()}


# Demonstration using first test patient from Fold 1
if "all_agents" in globals():
    demo_agent = all_agents[0]
    demo_patient = folds[0]["states_test"][0]
    result = predict_patient(demo_agent, demo_patient)

    print("\nInference Demo")
    print("-"*40)
    print("Recommended Action :", result["recommended_action"])
    print("First 10 Q-values  :", result["q_values"][:10])

    with open(os.path.join(OUTPUT_DIR, "demo_prediction.pkl"), "wb") as f:
        pickle.dump(result, f)
    print("Saved demo_prediction.pkl")
else:
    print("No trained agents available.")


Saved: deployment_outputs/evaluation_results.xlsx

Inference Demo
----------------------------------------
Recommended Action : 15
First 10 Q-values  : [-6.1879416 -6.3167467 -7.545349  -6.3438807 -5.909226  -6.0324473
 -6.12105   -6.1958427 -6.5827427 -6.203637 ]
Saved demo_prediction.pkl


### 22. COMPLETE EXPERIMENT SUMMARY & ARTIFACT EXPORT

In [ ]:
import json
import matplotlib.pyplot as plt

OUTPUT_DIR = "final_artifacts"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load evaluation results
results_path = "evaluation_results.csv"

if not os.path.exists(results_path):
    raise FileNotFoundError("evaluation_results.csv not found. "
                            "Run Block5_Part6_Evaluation.py first.")

results_df = pd.read_csv(results_path)

# Aggregate Metrics
summary = {"mean": results_df.mean(numeric_only=True).to_dict(),
           "std": results_df.std(numeric_only=True).to_dict(),
           "min": results_df.min(numeric_only=True).to_dict(),
           "max": results_df.max(numeric_only=True).to_dict()}

# Save JSON
json_path = os.path.join(OUTPUT_DIR, "experiment_summary.json")

with open(json_path, "w") as f:
    json.dump(summary, f, indent=4)

# Save Pickle
pickle_path = os.path.join(OUTPUT_DIR, "experiment_summary.pkl")

with open(pickle_path, "wb") as f:
    pickle.dump(summary, f)

# Markdown Report
md_path = os.path.join(OUTPUT_DIR, "experiment_report.md")
lines = []
lines.append("# EHR-DQN Experiment Report\n")
lines.append("## Average Metrics\n")

for k, v in summary["mean"].items():
    lines.append(f"- **{k}**: {v:.6f}")

lines.append("\n## Standard Deviation\n")

for k, v in summary["std"].items():
    lines.append(f"- **{k}**: {v:.6f}")

with open(md_path, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

# Bar Chart of Mean Metrics
mean_series = pd.Series(summary["mean"])

plt.figure(figsize=(8,5))
mean_series.plot(kind="bar")
plt.title("Average Evaluation Metrics")
plt.ylabel("Score")
plt.tight_layout()

plot_path = os.path.join(OUTPUT_DIR, "average_metrics.png")
plt.savefig(plot_path)
plt.close()

# Print Summary
print("\nAverage Metrics")
print(results_df.mean(numeric_only=True))

print("\nGenerated Files")
print("-"*40)
print(json_path)
print(pickle_path)
print(md_path)
print(plot_path)


Average Metrics
Fold           3.000000
Accuracy       0.499885
Precision      0.582706
Recall         0.550980
F1             0.533468
Specificity    0.413956
ROC_AUC        0.436518
dtype: float64

Generated Files
----------------------------------------
final_artifacts/experiment_summary.json
final_artifacts/experiment_summary.pkl
final_artifacts/experiment_report.md
final_artifacts/average_metrics.png


### 23. Export Best-Performing Fold Model (Highest Accuracy)

In [ ]:
# ==========================================================
# Select and Export the Best-Performing Fold Model
#
# Compares the per-fold Accuracy already computed in the evaluation cell
# above and exports ONLY that fold's model (not all 5) to a canonical
# best_model/ directory with a metadata.json, matching the artifact
# contract in models/README.md.
# ==========================================================
import shutil
import json
from datetime import datetime, timezone

BEST_MODEL_DIR = "best_model"
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

if "results_df" not in globals():
    raise RuntimeError("results_df not found. Run the evaluation cell above first.")

best_row = results_df.loc[results_df["Accuracy"].idxmax()]
best_fold = int(best_row["Fold"])
best_accuracy = float(best_row["Accuracy"])

print("=" * 70)
print("BEST FOLD SELECTION")
print("=" * 70)
print(results_df[["Fold", "Accuracy", "ROC_AUC"]].to_string(index=False))
print()
print(f"Best Fold      : {best_fold}")
print(f"Best Accuracy  : {best_accuracy:.4f}")

# Copy that fold's already-saved model (see "Save Models" cell above) to a
# canonical best-model path instead of shipping all 5 fold checkpoints.
source_model_path = os.path.join(MODEL_DIR, f"dqn_fold_{best_fold}.pth")
if not os.path.exists(source_model_path):
    raise FileNotFoundError(f"{source_model_path} not found. Run the 'Save Models' cell first.")

best_model_path = os.path.join(BEST_MODEL_DIR, "model.pth")
shutil.copyfile(source_model_path, best_model_path)
print(f"\nBest model copied to: {best_model_path}")

# Canonical feature order (see backend/utils/featureContract.js).
CANONICAL_FEATURE_ORDER = [
    "thalach", "restecg", "oldpeak", "slope", "age",
    "sex", "cp", "exang", "trestbps", "fbs"
]

metadata = {
    "model_version": f"dqn-fold{best_fold}-{datetime.now(timezone.utc).strftime('%Y-%m-%d')}",
    "algorithm": "DQN",
    "feature_order": CANONICAL_FEATURE_ORDER,
    "preprocessing": {
        "note": (
            f"This model was trained on a {STATE_SIZE}-dimension encoded state, "
            "NOT the 10 raw canonical features listed above. A preprocessing "
            "encoder mapping raw features to this training representation must "
            "be exported and validated before this model can be wired into the "
            "backend (see models/README.md, 'Known gap')."
        ),
        "trained_state_dimension": int(STATE_SIZE)
    },
    "training_metadata": {
        "fold": best_fold,
        "num_folds": len(folds),
        "episodes": NUM_EPISODES,
        "trained_on": datetime.now(timezone.utc).strftime("%Y-%m-%d")
    },
    "evaluation_metadata": {
        "accuracy": float(best_row["Accuracy"]),
        "precision": float(best_row["Precision"]),
        "recall": float(best_row["Recall"]),
        "f1": float(best_row["F1"]),
        "roc_auc": None if pd.isna(best_row["ROC_AUC"]) else float(best_row["ROC_AUC"])
    },
    "runtime_compatibility": {
        "framework": "pytorch",
        "min_python": "3.10"
    }
}

metadata_path = os.path.join(BEST_MODEL_DIR, "metadata.json")
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Metadata saved to: {metadata_path}")

# Trigger a browser download when running in Google Colab; otherwise just
# report the local path (e.g. when run on a local Jupyter kernel).
try:
    from google.colab import files
    files.download(best_model_path)
    files.download(metadata_path)
    print("\nDownload triggered for best_model/model.pth and metadata.json")
except ImportError:
    print("\nNot running in Google Colab -- files are available locally at:")
    print(f"  {best_model_path}")
    print(f"  {metadata_path}")
print("=" * 70)

### 24. Export to ONNX (Node-Loadable Format for the Backend)

In [ ]:
# ==========================================================
# Export the Best-Performing Fold Model to ONNX
#
# The backend (Node.js) cannot load a PyTorch .pth checkpoint directly.
# ONNX is an open, framework-neutral format the backend's onnxAdapter.js
# can run via onnxruntime-node. This exports the SAME best-fold model
# selected in the previous cell, in addition to (not instead of) model.pth.
#
# IMPORTANT: onnxruntime-node (the backend's ONNX runtime) only supports
# ONNX IR version <= 10. torch.onnx.export() normally produces a compatible
# IR version automatically -- if you see "Unsupported model IR version"
# when the backend loads this file, re-export with a lower opset_version.
# ==========================================================
import torch

best_policy_network = DeepQNetwork(STATE_SIZE, ACTION_SIZE).to(device)
best_policy_network.load_state_dict(torch.load(best_model_path, map_location=device))
best_policy_network.eval()

onnx_path = os.path.join(BEST_MODEL_DIR, "model.onnx")
dummy_input = torch.randn(1, STATE_SIZE, device=device)

torch.onnx.export(
    best_policy_network,
    dummy_input,
    onnx_path,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
    opset_version=13,
)

print(f"ONNX model exported to: {onnx_path}")

# Record how the backend should interpret this model's output and which
# input tensor name to feed (see models/README.md, "runtime_compatibility").
# DQN's raw output has ACTION_SIZE = NUM_FEATURES * 3 values (a
# feature-modification action space), but this notebook's own evaluation
# cell above only uses the FIRST TWO as a binary classifier signal
# (q_values[:, :2]) -- num_classes tells the backend to do the same.
metadata["runtime_compatibility"]["export_format"] = "onnx"
metadata["runtime_compatibility"]["input_name"] = "input"
metadata["runtime_compatibility"]["num_classes"] = 2

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)
print(f"Updated metadata.json with ONNX runtime_compatibility fields.")

try:
    from google.colab import files
    files.download(onnx_path)
    files.download(metadata_path)
    print("\nDownload triggered for best_model/model.onnx and updated metadata.json")
except ImportError:
    print(f"\nNot running in Google Colab -- file available locally at: {onnx_path}")